# 03 - Pelatihan & Evaluasi Model

Fungsi `run()` melakukan: latih satu konfigurasi (model x regime x seed), pilih
checkpoint terbaik dari **val kondisi latih**, lalu evaluasi pada SEMUA kondisi uji
tanpa pelatihan ulang. Prosedur identik untuk MobileNetV2 dan ResNet50.

## 1. Setup

In [ ]:
import os, sys, json
from pathlib import Path
from IPython.display import display, Image

# Temukan root proyek (folder berisi configs/experiment.yaml), lalu jadikan cwd.
_cands = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
ROOT = next(p for p in _cands if (p / "configs" / "experiment.yaml").exists())
os.chdir(ROOT)
for _p in (ROOT / "src", ROOT / "scripts"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from vehicle_lighting.config import load_config, resolve
cfg = load_config()
print("ROOT   :", ROOT)
print("config :", ROOT / "configs" / "experiment.yaml")
print("kelas  :", cfg["data"]["classes"])
print("model  :", cfg["models"])

In [ ]:
from vehicle_lighting.train import run
results_dir = resolve(cfg, cfg["results_dir"])

## 2. Smoke test (1 epoch, tanpa bobot pretrained)

Pastikan pipeline berjalan sebelum menjalankan eksperimen penuh.

In [ ]:
_ = run(cfg, "mobilenet_v2", "normal", cfg["project"]["seeds"][0], epochs=1, pretrained=False)

## 3. Eksperimen utama

2 model x 2 regime x 3 seed. Run yang sudah punya `metrics.json` akan dilewati,
sehingga cell ini aman dijalankan ulang bila terputus.

In [ ]:
REGIMES = ["normal", "mixed"]      # tambah normal_brightaug / normal_gamma / normal_clahe bila perlu
MODELS = cfg["models"]
SEEDS = cfg["project"]["seeds"]
EPOCHS = None                      # None = pakai nilai di config
NO_PRETRAINED = False

for reg in REGIMES:
    for m in MODELS:
        for s in SEEDS:
            done = results_dir / f"{m}__{reg}__seed{s}" / "metrics.json"
            if done.exists():
                print("lewati (sudah ada):", done.parent.name)
                continue
            print("==>", m, reg, "seed", s, flush=True)
            run(cfg, m, reg, s, EPOCHS, False if NO_PRETRAINED else None, save_ckpt=True)

Catatan: tanpa `pretrained=False`, bobot ImageNet diunduh sekali pada run pertama
(butuh internet). Lanjut ke `04_evaluasi_visualisasi.ipynb`.